# TranSTR + DINOv3 + groundingDINO + NCOD LUM+HUM — Colab Pro
**Khác bản Kaggle**: data qua `kaggle_hub`, bs=32, no gradient accumulation, 10 epochs, EMA weights cho eval/save, tắt runtime cell cuối.

In [ ]:
# CELL 0: Mount Google Drive (lưu checkpoint)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CELL 1: Git Clone & Setup
import os
REPO_URL  = 'https://github.com/DanielQH07/tranSTR_Casual.git'
REPO_NAME = 'tranSTR_Casual'
BRANCH    = 'full_mode'

if not os.path.exists(REPO_NAME):
    print(f'Cloning {REPO_URL}...')
    os.system(f'git clone {REPO_URL} -b {BRANCH}')
else:
    print('Repo already exists.')

target_dir = os.path.join(os.getcwd(), REPO_NAME, 'causalvid')
if os.path.exists(target_dir) and os.path.basename(os.getcwd()) != 'causalvid':
    os.chdir(target_dir)
    print(f'Changed directory to: {os.getcwd()}')
else:
    print(f'Working directory: {os.getcwd()}')

Cloning https://github.com/DanielQH07/tranSTR_Casual.git...
Working directory: /content/tranSTR_Casual


In [ ]:
%cd /content/tranSTR_Casual

/content/tranSTR_Casual


In [ ]:
# CELL 2: Install dependencies & W&B login
print('=== CELL 2: Install & W&B Setup ===')
os.system('pip install -q wandb kaggle kaggle-hub --upgrade')

import wandb

# ============================================
# W&B CONFIG
# ============================================
WANDB_API_KEY = ''  # 🔴 API key
WANDB_PROJECT = 'transtr-causalvid-dino'
WANDB_ENTITY  = None

wandb.login(key=WANDB_API_KEY)
print('✅ W&B logged in!')

=== CELL 2: Install & W&B Setup ===


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


✅ W&B logged in!


In [ ]:
# !mkdir -p ~/.kaggle
# !cp /content/drive/MyDrive/KLTN/kaggle.json ~/.kaggle/

In [ ]:
# CELL 3: Setup Kaggle API key & download datasets via kagglehub
print('=== CELL 3: Kaggle Hub Data Download ===')

import os
import json
import glob
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Ensure required packages are available in Colab
!pip install -q -U kagglehub kaggle wandb

# ============================================
# KAGGLE API CREDENTIALS
# Option A: Colab Secrets (recommended)
# Option B: Local kaggle.json fallback
# ============================================
creds_ok = False
try:
    from google.colab import userdata
    username = userdata.get('KAGGLE_USERNAME')
    key = userdata.get('KAGGLE_KEY')
    if username and key:
        os.environ['KAGGLE_USERNAME'] = username
        os.environ['KAGGLE_KEY'] = key
        creds_ok = True
        print('Kaggle credentials loaded from Colab Secrets')
except Exception as e:
    print(f'Could not load from Colab Secrets: {e}')

if not creds_ok:
    kaggle_json_candidates = [
        '/content/kaggle.json',
        '/content/.kaggle/kaggle.json',
        os.path.expanduser('~/.kaggle/kaggle.json'),
        '/content/drive/MyDrive/KLTN/kaggle.json'
    ]
    found = None
    for p in kaggle_json_candidates:
        if os.path.exists(p):
            found = p
            break

    if found is None:
        raise FileNotFoundError(
            'Kaggle credentials not found. Add KAGGLE_USERNAME/KAGGLE_KEY in Colab Secrets or upload kaggle.json.'
        )

    with open(found, 'r', encoding='utf-8') as f:
        kg = json.load(f)
    os.environ['KAGGLE_USERNAME'] = kg.get('username', '')
    os.environ['KAGGLE_KEY'] = kg.get('key', '')
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp {found} /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json
    print(f'Kaggle credentials loaded from: {found}')

import kagglehub

def _find_best_dir(root_path, preferred_dir_names):
    root = Path(root_path)
    if not root.exists():
        return None

    # 1) Exact directory-name match (deep search)
    for name in preferred_dir_names:
        exact_matches = [p for p in root.rglob('*') if p.is_dir() and p.name == name]
        if exact_matches:
            return str(exact_matches[0])

    # 2) Directory containing many .npy files
    npy_counts = {}
    for p in root.rglob('*.npy'):
        parent = str(p.parent)
        npy_counts[parent] = npy_counts.get(parent, 0) + 1
    if npy_counts:
        return max(npy_counts.items(), key=lambda x: x[1])[0]

    # 3) Directory containing many .pt files
    pt_counts = {}
    for p in root.rglob('*.pt'):
        parent = str(p.parent)
        pt_counts[parent] = pt_counts.get(parent, 0) + 1
    if pt_counts:
        return max(pt_counts.items(), key=lambda x: x[1])[0]

    # 4) Fallback root
    return str(root)

# ============================================
# Dataset slugs (GroundingDINO full + DINOv3 + annotations + splits)
# ============================================
print('\nDownloading datasets (uses cache if available)...')

DINOV3_HUB_PATH = kagglehub.dataset_download('danielq07/dinov3-feat')
print(f'DINOv3 features root: {DINOV3_HUB_PATH}')

# GroundingDINO full features (ROI 1024-d + class_text_embedding 768-d)
GDINO_HUB_PATH = kagglehub.dataset_download('danielq07/GroundingDINO-full-feature')
print(f'GroundingDINO full features root: {GDINO_HUB_PATH}')

# Handle zip: if dataset contains a zip instead of loose pkl files, extract it
import zipfile
from collections import Counter as _Counter
_gdino_pkls = glob.glob(os.path.join(GDINO_HUB_PATH, '**', '*.pkl'), recursive=True)
if _gdino_pkls:
    _dir_counts = _Counter(os.path.dirname(f) for f in _gdino_pkls)
    _GDINO_PKL_DIR = _dir_counts.most_common(1)[0][0]
    print(f'Found {len(_gdino_pkls)} pkl files in: {_GDINO_PKL_DIR}')
else:
    _zip_files = glob.glob(os.path.join(GDINO_HUB_PATH, '**', '*.zip'), recursive=True)
    if not _zip_files:
        raise FileNotFoundError(f'No pkl or zip files found in {GDINO_HUB_PATH}')
    _GDINO_PKL_DIR = '/content/gdino_full_features' if os.path.exists('/content') else '/kaggle/working/gdino_full_features'
    os.makedirs(_GDINO_PKL_DIR, exist_ok=True)
    for _zf in _zip_files:
        print(f'Extracting {_zf}...')
        with zipfile.ZipFile(_zf, 'r') as z:
            z.extractall(_GDINO_PKL_DIR)
    _gdino_pkls = glob.glob(os.path.join(_GDINO_PKL_DIR, '**', '*.pkl'), recursive=True)
    if _gdino_pkls:
        _dir_counts = _Counter(os.path.dirname(f) for f in _gdino_pkls)
        _GDINO_PKL_DIR = _dir_counts.most_common(1)[0][0]
    print(f'Extracted {len(_gdino_pkls)} pkl files to: {_GDINO_PKL_DIR}')

ANN_HUB_PATH = kagglehub.dataset_download('lusnaw/text-annotation')
print(f'Annotations root: {ANN_HUB_PATH}')

SPLIT_HUB_PATH = kagglehub.dataset_download('danielq07/casual-vid-data-split')
print(f'Splits root: {SPLIT_HUB_PATH}')

# ============================================
# Merge DINOv3 features (train/val/test -> merged folder)
# ============================================
print('\n=== Merge DINOv3 Features ===')
CLIP_MERGED_PATH = '/content/dinov3_T16_dim1024_merge' if os.path.exists('/content') else '/kaggle/working/dinov3_T16_dim1024_merge'
os.makedirs(CLIP_MERGED_PATH, exist_ok=True)

candidate_roots = [
    DINOV3_HUB_PATH,
    os.path.join(DINOV3_HUB_PATH, 'features'),
]

def has_split_subdirs(root):
    if not os.path.exists(root):
        return False
    names = set(os.listdir(root))
    return ('train' in names) and ('test' in names) and ('valid' in names or 'val' in names)

resolved_source = None
for cand in candidate_roots:
    if has_split_subdirs(cand):
        resolved_source = cand
        break

if resolved_source is None:
    # Fallback: if dataset is already merged in a .pt directory, reuse it
    maybe_merged = _find_best_dir(DINOV3_HUB_PATH, ['dinov3_T16_dim1024_merge', 'dinov3-feat'])
    pt_files = [f for f in os.listdir(maybe_merged)] if (maybe_merged and os.path.exists(maybe_merged)) else []
    pt_files = [f for f in pt_files if f.endswith('.pt')]
    if len(pt_files) > 0:
        CLIP_MERGED_PATH = maybe_merged
        print(f'DINOv3 already merged, reuse path: {CLIP_MERGED_PATH} ({len(pt_files)} .pt)')
    else:
        raise FileNotFoundError(f'Cannot find DINOv3 split folders. Tried: {candidate_roots}')
else:
    print(f'DINOv3 split source: {resolved_source}')
    total_copied = 0
    for split in ['train', 'test', 'valid', 'val']:
        split_folder = os.path.join(resolved_source, split)
        if not os.path.exists(split_folder):
            continue
        split_pt_files = [f for f in os.listdir(split_folder) if f.endswith('.pt')]
        print(f'{split}: {len(split_pt_files)} files')
        for fname in tqdm(split_pt_files, desc=f'Copying {split}'):
            src = os.path.join(split_folder, fname)
            dst = os.path.join(CLIP_MERGED_PATH, fname)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                total_copied += 1
    final_count = len([f for f in os.listdir(CLIP_MERGED_PATH) if f.endswith('.pt')])
    print(f'Merge complete: {final_count} .pt (copied {total_copied} new files)')

# Resolved paths for downstream cells
GDINO_MERGED_PATH = _GDINO_PKL_DIR
ANNOTATION_QA_PATH = _find_best_dir(ANN_HUB_PATH, ['QA'])
SPLIT_TXT_PATH = _find_best_dir(SPLIT_HUB_PATH, ['split'])

print('\nResolved training paths:')
print(f'CLIP_MERGED_PATH: {CLIP_MERGED_PATH}')
print(f'GDINO_MERGED_PATH: {GDINO_MERGED_PATH}')
print(f'ANNOTATION_QA_PATH: {ANNOTATION_QA_PATH}')
print(f'SPLIT_TXT_PATH: {SPLIT_TXT_PATH}')

print('Kaggle Hub download done.')

=== CELL 3: Kaggle Hub Data Download ===
Kaggle credentials loaded from Colab Secrets

Using Colab cache for faster access to the 'dinov3-feat' dataset.
DINOv3 features root: /kaggle/input/dinov3-feat
Using Colab cache for faster access to the 'gdino-roi-all-nodes-merged' dataset.
GroundingDINO ROI features root: /kaggle/input/gdino-roi-all-nodes-merged
Using Colab cache for faster access to the 'text-annotation' dataset.
Annotations root: /kaggle/input/text-annotation
Splits root: /root/.cache/kagglehub/datasets/danielq07/casual-vid-data-split/versions/1

=== Merge DINOv3 Features ===
DINOv3 split source: /kaggle/input/dinov3-feat/features
train: 18776 files


Copying train:   0%|          | 0/18776 [00:00<?, ?it/s]

test: 5429 files


Copying test:   0%|          | 0/5429 [00:00<?, ?it/s]

valid: 2695 files


Copying valid:   0%|          | 0/2695 [00:00<?, ?it/s]

Merge complete: 26900 .pt (copied 0 new files)

Resolved training paths:
CLIP_MERGED_PATH: /content/dinov3_T16_dim1024_merge
GDINO_MERGED_PATH: /kaggle/input/gdino-roi-all-nodes-merged
ANNOTATION_QA_PATH: /kaggle/input/text-annotation/QA
SPLIT_TXT_PATH: /root/.cache/kagglehub/datasets/danielq07/casual-vid-data-split/versions/1/split
Kaggle Hub download done.


In [ ]:
# CELL 4: Core imports + Qwen training functions
print('=== CELL 4: Imports + Functions (MiniLM + Qwen) ===')

# Core training imports required by CELL 5 onward
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm

from utils.util import set_seed, set_gpu_devices
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel

def _unpack_batch(batch):
    if len(batch) == 7:
        ff, of, q, a, ans_id, qns_key, q_family_id = batch
    elif len(batch) == 6:
        ff, of, q, a, ans_id, qns_key = batch
        q_family_id = None
    else:
        raise ValueError(f'Unexpected batch format with {len(batch)} elements')
    return ff, of, q, a, ans_id, qns_key, q_family_id

def train_epoch_qwen(
    model, optimizer, loader, ce_loss, device, epoch, accumulation_steps=1
 ):
    model.train()
    total_loss = 0.0
    correct, total = 0, 0
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    for batch_idx, batch in enumerate(pbar):
        ff, of, q, a, ans_id, _, q_family_id = _unpack_batch(batch)
        ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
        q_family_id = q_family_id.to(device) if q_family_id is not None else None

        out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
        logits = out['qwen_logits']
        loss = ce_loss(logits, tgt)
        (loss / accumulation_steps).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item()
        correct += (logits.argmax(-1) == tgt).sum().item()
        total += tgt.size(0)

        pbar.set_postfix({
            'loss': total_loss / (batch_idx + 1),
            'acc': correct / max(total, 1) * 100
        })

    n = len(loader)
    return total_loss / n, correct / max(total, 1) * 100

def eval_epoch_qwen(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            ff, of, q, a, ans_id, _, q_family_id = _unpack_batch(batch)
            ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
            logits = out['qwen_logits']
            correct += (logits.argmax(-1) == tgt).sum().item()
            total += tgt.size(0)
    return correct / max(total, 1) * 100

print('Imports and functions defined for MiniLM + Qwen training.')

=== CELL 4: Imports + Functions (NCOD + HUM + Verifier/Knowledge) ===
Imports and functions defined for integrated training.


In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Imports OK')

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM: 102.0 GB
Imports OK


In [ ]:
# CELL 5: Setup Paths & Config
print('=== CELL 5: Paths & Config ===')

# ============================================
# DATA PATHS (prefer kagglehub paths resolved in CELL 3)
# ============================================
clip_merged_path = globals().get('CLIP_MERGED_PATH', None)
gdino_merged_path = globals().get('GDINO_MERGED_PATH', None)
annotation_qa_path = globals().get('ANNOTATION_QA_PATH', None)
split_txt_path = globals().get('SPLIT_TXT_PATH', None)

if clip_merged_path:
    CLIP_FEATURE_PATH = clip_merged_path
else:
    CLIP_FEATURE_PATH = '/kaggle/working/dinov3_T16_dim1024_merge'

if gdino_merged_path:
    GDINO_FEATURE_PATH = gdino_merged_path
else:
    GDINO_FEATURE_PATH = '/kaggle/input/datasets/danielq07/GroundingDINO-full-feature'

if annotation_qa_path:
    ANNOTATION_PATH = annotation_qa_path
else:
    ANNOTATION_PATH = '/kaggle/input/text-annotation/QA'

if split_txt_path:
    SPLIT_DIR = split_txt_path
else:
    SPLIT_DIR = '/kaggle/input/casual-vid-data-split/split'

# Working directories
BASE = '/content/working' if os.path.exists('/content') else '/kaggle/working'
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# Verify paths
print('\n--- Path Verification ---')
def verify_path(name, path):
    if os.path.exists(path):
        items = os.listdir(path)[:3]
        print(f'OK {name}: {items}')
        return True
    print(f'NOT FOUND {name}: {path}')
    return False

all_ok = True
all_ok &= verify_path('DINOv3 Features (1024D)', CLIP_FEATURE_PATH)
all_ok &= verify_path('GroundingDINO Full Features', GDINO_FEATURE_PATH)
all_ok &= verify_path('Annotations (QA)', ANNOTATION_PATH)
all_ok &= verify_path('Splits', SPLIT_DIR)

if not all_ok:
    raise FileNotFoundError(
        'One or more required data paths are missing. Re-run CELL 3 and check resolved kagglehub paths.'
    )

# ============================================
# RUN PRESET (MiniLM + Qwen2.5-VL)
# ============================================
RUN_TRAINING = True
RUN_PROFILE = 'run1'  # ['run1', 'run2']

RUN_PROFILES = {
    'run1': {
        'epoch': 8,
        'lr_main': 1e-4,
        'lr_qwen': 5e-5,
        'early_stop_start_epoch': 4,
        'early_stop_patience': 3,
        'dropout': 0.3,
        'encoder_dropout': 0.3,
        'decay': 1e-4,
    },
    'run2': {
        'epoch': 10,
        'lr_main': 8e-5,
        'lr_qwen': 4e-5,
        'early_stop_start_epoch': 5,
        'early_stop_patience': 4,
        'dropout': 0.3,
        'encoder_dropout': 0.3,
        'decay': 1e-4,
    },
}

if RUN_PROFILE not in RUN_PROFILES:
    raise ValueError(f'Invalid RUN_PROFILE={RUN_PROFILE}. Must be one of {list(RUN_PROFILES.keys())}')

RUN_TAG = RUN_PROFILE
MODEL_FILENAME = f'best_model_qwen_minilm_{RUN_TAG}.ckpt'
LATEST_CKPT_FILENAME = f'latest_checkpoint_qwen_minilm_{RUN_TAG}.ckpt'
TRAIN_HISTORY_FILENAME = f'train_history_qwen_minilm_{RUN_TAG}.csv'
PREDICTIONS_CSV_FILENAME = f'test_predictions_qwen_minilm_{RUN_TAG}.csv'
METRICS_JSON_FILENAME = f'final_metrics_qwen_minilm_{RUN_TAG}.json'
BEST_ARTIFACT_NAME = f'best-model-qwen-minilm-{RUN_TAG}'
LAST_ARTIFACT_NAME = f'last-checkpoint-qwen-minilm-{RUN_TAG}'
FINAL_ARTIFACT_NAME = f'final-results-qwen-minilm-{RUN_TAG}'

FEAT_DIM = 1024
print(f'\nBackbone: DINOv3 (D={FEAT_DIM}) + GroundingDINO Full (ROI 1024 + cls_emb 768 + bbox 4 = 1796)')
print(f'Run profile: {RUN_PROFILE}')

class Config:
    # Paths
    video_feature_root = CLIP_FEATURE_PATH
    grounding_dino_path = GDINO_FEATURE_PATH
    sample_list_path = ANNOTATION_PATH
    split_dir_txt = SPLIT_DIR

    # Model architecture
    topK_frame = 16
    objs = 20
    frames = 16
    select_frames = 5
    topK_obj = 12

    frame_feat_dim = FEAT_DIM
    obj_feat_dim = 1796
    use_grounding_dino = True

    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3

    # MiniLM for question-guided frame selection
    minilm_name = 'sentence-transformers/all-MiniLM-L6-v2'
    freeze_minilm = True

    # Qwen2.5-VL
    qwen_name = 'Qwen/Qwen2.5-VL-3B-Instruct'
    use_qwen_lora = True
    qwen_lora_r = 8
    qwen_lora_alpha = 16
    qwen_lora_dropout = 0.1
    qwen_lora_target_modules = None
    evidence_top_m = 5
    use_qwen_causal_filter = False
    qwen_max_new_tokens = 64

    # Training
    bs = 16
    accumulation_steps = 1
    lr_main = 1e-4
    lr_qwen = 5e-5
    epoch = 8
    gpu = 0
    gamma = 0.5
    decay = 1e-4
    n_query = 5
    return_family_id = True

    # LR Scheduler + Early stopping
    lr_patience = 1
    min_lr = 1e-7
    early_stop_patience = 3
    early_stop_min_delta = 0.05
    early_stop_start_epoch = 4

    # Other
    hard_eval = False
    pos_ratio = 1.0
    neg_ratio = 1.0
    a = 1.0
    num_workers = 4

# Apply selected preset overrides
for _k, _v in RUN_PROFILES[RUN_PROFILE].items():
    setattr(Config, _k, _v)

args = Config()

set_gpu_devices(args.gpu)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
print(f'Run tag: {RUN_TAG}')
print(f'Config: frame_feat_dim={args.frame_feat_dim}, obj_feat_dim={args.obj_feat_dim}')
print(f'Using GroundingDINO features: {args.use_grounding_dino}')
print(f'Gradient Accumulation: physical_bs={args.bs} x {args.accumulation_steps} = effective_bs={args.bs * args.accumulation_steps}')
print(f'Learning rate: main={args.lr_main} | qwen={args.lr_qwen}')
print(f'Weight decay: {args.decay}')
print(f'Early stopping: start_epoch={args.early_stop_start_epoch}, patience={args.early_stop_patience}, min_delta={args.early_stop_min_delta}')
print(f'Model filename: {MODEL_FILENAME}')

=== CELL 5: Paths & Config ===

--- Path Verification ---
OK DINOv3 Features (1024D): ['QVpSZ1yi1EQ_000040_000050.pt', 'mj3wCvbeWEg_000048_000058.pt', 'GXf1yeGWMMg_000018_000028.pt']
OK GroundingDINO ROI Features: ['YEI7nmkbgQA_000040_000050.pkl', 'E5y8S1dOfkQ_000003_000013.pkl', 'dtgFHqPHnCo_000050_000060.pkl']
OK Annotations (QA): ['P-JmNb-FcLk_000041_000051', '2VBmRPrfNZY_000000_000010', '_muKn2ZeLK0_000018_000028']
OK Splits: ['train.pkl', 'test.pkl', 'valid.pkl']

Backbone: DINOv3 (D=1024) + GroundingDINO ROI

Device: cuda
Config: frame_feat_dim=1024, obj_feat_dim=1028
Using GroundingDINO ROI features: True
Gradient Accumulation: physical_bs=32 x 1 = effective_bs=32
Verifier loss weight lambda: 0.3
Knowledge loss weight lambda: 0.2
Return question family id: True
Full text-encoder fine-tuning: True
LoRA enabled: False
Early stopping: start_epoch=5, patience=4, min_delta=0.05
LR scheduler: factor=0.5, patience=1, min_lr=1e-07


In [ ]:
# CELL 6: Create Datasets
print('=== CELL 6: Datasets ===')

train_ds = VideoQADataset(
    split='train', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
val_ds = VideoQADataset(
    split='val', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
test_ds = VideoQADataset(
    split='test', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)

train_loader = DataLoader(train_ds, args.bs, shuffle=True, num_workers=args.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=True)

train_sample_keys = [f"{row.video_id}_{row.type}" for row in train_ds.sample_list.itertuples(index=False)]
train_key_to_idx = {k: i for i, k in enumerate(train_sample_keys)}

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

=== CELL 6: Datasets ===
[train] Object feature format: unknown
[train] Using GroundingDINO features
[train] Indexed 26884 object features, 26900 ViT features
[train] Loaded 18776 video IDs from /root/.cache/kagglehub/datasets/danielq07/casual-vid-data-split/versions/1/split/train.pkl
[train] ViT: 18776, Obj: 18765, Both: 18765


[train] Parsing annotations:   0%|          | 0/18765 [00:00<?, ?it/s]

[train] Final: 112590 QA pairs
[val] Object feature format: unknown
[val] Using GroundingDINO features
[val] Indexed 26884 object features, 26900 ViT features
[val] Loaded 2695 video IDs from /root/.cache/kagglehub/datasets/danielq07/casual-vid-data-split/versions/1/split/valid.pkl
[val] ViT: 2695, Obj: 2693, Both: 2693


[val] Parsing annotations:   0%|          | 0/2693 [00:00<?, ?it/s]

[val] Final: 16158 QA pairs
[test] Object feature format: unknown
[test] Using GroundingDINO features
[test] Indexed 26884 object features, 26900 ViT features
[test] Loaded 5429 video IDs from /root/.cache/kagglehub/datasets/danielq07/casual-vid-data-split/versions/1/split/test.pkl
[test] ViT: 5429, Obj: 5426, Both: 5426


[test] Parsing annotations:   0%|          | 0/5426 [00:00<?, ?it/s]

[test] Final: 32556 QA pairs
Train: 112590 | Val: 16158 | Test: 32556


In [ ]:
# CELL 7: Model + Optimizer (MiniLM + Qwen LoRA)
print('=== CELL 7: Model ===')
cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames
model = VideoQAmodel(**cfg)
model.to(device)

qwen_params = []
other_params = []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith('qwen.'):
        qwen_params.append(p)
    else:
        other_params.append(p)

param_groups = []
if other_params:
    param_groups.append({'params': other_params, 'lr': args.lr_main, 'weight_decay': args.decay})
if qwen_params:
    param_groups.append({'params': qwen_params, 'lr': args.lr_qwen, 'weight_decay': args.decay})
if not param_groups:
    raise RuntimeError('No trainable parameters found for optimizer_model.')

optimizer_model = torch.optim.AdamW(param_groups)
scheduler = ReduceLROnPlateau(
    optimizer_model,
    mode='max',
    factor=args.gamma,
    patience=args.lr_patience,
    threshold=args.early_stop_min_delta,
    threshold_mode='abs',
    min_lr=args.min_lr
)

ce_loss = nn.CrossEntropyLoss()
save_path = os.path.join(MODEL_DIR, MODEL_FILENAME)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable/1e6:.2f}M')
print(f'Qwen trainable params: {sum(p.numel() for p in qwen_params)/1e6:.2f}M')
print(f'Artifacts: best={BEST_ARTIFACT_NAME} | latest={LAST_ARTIFACT_NAME}')

=== CELL 7: Model ===


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaModel LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable params: 218.97M
Text-encoder trainable params: 138.602M
U shape: (112590,)


In [ ]:
# CELL 7: Model + Optimizer (MiniLM + Qwen LoRA)
print('=== CELL 7: Model ===')
cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames
model = VideoQAmodel(**cfg)
model.to(device)

qwen_params = []
other_params = []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith('qwen.'):
        qwen_params.append(p)
    else:
        other_params.append(p)

param_groups = []
if other_params:
    param_groups.append({'params': other_params, 'lr': args.lr_main, 'weight_decay': args.decay})
if qwen_params:
    param_groups.append({'params': qwen_params, 'lr': args.lr_qwen, 'weight_decay': args.decay})
if not param_groups:
    raise RuntimeError('No trainable parameters found for optimizer_model.')

optimizer_model = torch.optim.AdamW(param_groups)
scheduler = ReduceLROnPlateau(
    optimizer_model,
    mode='max',
    factor=args.gamma,
    patience=args.lr_patience,
    threshold=args.early_stop_min_delta,
    threshold_mode='abs',
    min_lr=args.min_lr
)

ce_loss = nn.CrossEntropyLoss()
save_path = os.path.join(MODEL_DIR, MODEL_FILENAME)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable/1e6:.2f}M')
print(f'Qwen trainable params: {sum(p.numel() for p in qwen_params)/1e6:.2f}M')
print(f'Artifacts: best={BEST_ARTIFACT_NAME} | latest={LAST_ARTIFACT_NAME}')

=== CELL 7: Model ===


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaModel LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable params: 218.97M
Text-encoder trainable params: 138.602M
U shape: (112590,)


In [ ]:
# CELL 8: Init W&B + Resume Checkpoint
print('=== CELL 8: Initialize W&B Run ===')

start_epoch = 1
best_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
history_rows = []

LATEST_CKPT_PATH = os.path.join(MODEL_DIR, LATEST_CKPT_FILENAME)
TRAIN_HISTORY_CSV_PATH = os.path.join(MODEL_DIR, TRAIN_HISTORY_FILENAME)

# For clean tuning, default to fresh training.
# Set True only if you intentionally continue the same RUN_TAG.
RESUME_FROM_CHECKPOINT = False
RESUME_SOURCE = 'wandb'  # 'local' or 'wandb'
RESUME_ARTIFACT_ALIAS = 'latest'
LOCAL_RESUME_PATH = LATEST_CKPT_PATH

wandb_config = {
    'run_tag': RUN_TAG,
    'run_profile': RUN_PROFILE,
    'backbone': 'dinov3+groundingdino',
    'minilm_name': args.minilm_name,
    'qwen_name': args.qwen_name,
    'use_qwen_lora': args.use_qwen_lora,
    'physical_batch_size': args.bs,
    'accumulation_steps': args.accumulation_steps,
    'effective_batch_size': args.bs * args.accumulation_steps,
    'epochs': args.epoch,
    'evidence_top_m': args.evidence_top_m,
    'lr_main': args.lr_main,
    'lr_qwen': args.lr_qwen,
    'lr_scheduler_factor': args.gamma,
    'lr_scheduler_patience': args.lr_patience,
    'min_lr': args.min_lr,
    'early_stop_patience': args.early_stop_patience,
    'early_stop_min_delta': args.early_stop_min_delta,
    'early_stop_start_epoch': args.early_stop_start_epoch,
    'resume_enabled': RESUME_FROM_CHECKPOINT,
    'resume_source': RESUME_SOURCE
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, config=wandb_config, reinit=True)
wandb.watch(model, log='gradients', log_freq=100)
print(f'W&B run: {run.url}')

def _load_resume_checkpoint(path, map_location):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Checkpoint not found: {path}')
    return torch.load(path, map_location=map_location)

if RESUME_FROM_CHECKPOINT:
    print(f'Resume enabled from: {RESUME_SOURCE}')
    try:
        checkpoint = None
        resume_path = None

        if RESUME_SOURCE == 'local':
            resume_path = LOCAL_RESUME_PATH
            checkpoint = _load_resume_checkpoint(resume_path, device)

        elif RESUME_SOURCE == 'wandb':
            api = wandb.Api()
            resume_entity = WANDB_ENTITY or api.default_entity
            artifact_path = f'{resume_entity}/{WANDB_PROJECT}/{LAST_ARTIFACT_NAME}:{RESUME_ARTIFACT_ALIAS}'
            print(f'Downloading artifact: {artifact_path}')
            artifact = api.artifact(artifact_path)
            artifact_dir = artifact.download()
            candidate_path = os.path.join(artifact_dir, LATEST_CKPT_FILENAME)
            if os.path.exists(candidate_path):
                resume_path = candidate_path
            else:
                ckpt_files = [f for f in os.listdir(artifact_dir) if f.endswith('.ckpt')]
                if not ckpt_files:
                    raise FileNotFoundError(f'No .ckpt found in artifact folder: {artifact_dir}')
                resume_path = os.path.join(artifact_dir, ckpt_files[0])
            checkpoint = _load_resume_checkpoint(resume_path, device)

        else:
            raise ValueError("RESUME_SOURCE must be 'local' or 'wandb'")

        ckpt_state = checkpoint['model_state_dict']
        model_state = model.state_dict()
        filtered_state = {}
        skipped_keys = []
        for k, v in ckpt_state.items():
            if k in model_state and v.shape != model_state[k].shape:
                skipped_keys.append(f'{k}: ckpt={list(v.shape)} vs model={list(model_state[k].shape)}')
            else:
                filtered_state[k] = v

        if skipped_keys:
            print(f'⚠️ Skipped {len(skipped_keys)} keys due to shape mismatch:')
            for sk in skipped_keys:
                print(f'  {sk}')
            print('These layers will be randomly initialized. Training from epoch 1.')

        missing, unexpected = model.load_state_dict(filtered_state, strict=False)
        if missing:
            print(f'Warning: missing model keys when resume: {len(missing)}')
        if unexpected:
            print(f'Warning: unexpected model keys when resume: {len(unexpected)}')

        if not skipped_keys:
            if 'optimizer_model_state_dict' in checkpoint:
                optimizer_model.load_state_dict(checkpoint['optimizer_model_state_dict'])
            if 'scheduler_state_dict' in checkpoint:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

            best_acc = float(checkpoint.get('best_acc', 0.0))
            start_epoch = int(checkpoint.get('epoch', 0)) + 1
            best_epoch = int(checkpoint.get('best_epoch', 0))
            epochs_without_improvement = int(checkpoint.get('epochs_without_improvement', 0))
            history = checkpoint.get('history', history)
            history_rows = checkpoint.get('history_rows', history_rows)

            if os.path.exists(TRAIN_HISTORY_CSV_PATH):
                try:
                    history_rows = pd.read_csv(TRAIN_HISTORY_CSV_PATH).to_dict('records')
                    print(f'Loaded history CSV with {len(history_rows)} rows')
                except Exception as csv_err:
                    print(f'Warning: failed to load history CSV: {csv_err}')
        else:
            print('Optimizer/scheduler NOT restored due to architecture change. Starting fresh training state.')

        print(f'Resumed from: {resume_path}')
        print(f'Start epoch: {start_epoch} | Best acc: {best_acc:.2f}% | Best epoch: {best_epoch}')
        wandb.run.summary['resume_path'] = str(resume_path)
        wandb.run.summary['resume_start_epoch'] = int(start_epoch)
        wandb.run.summary['resume_best_acc'] = float(best_acc)
        wandb.run.summary['resume_best_epoch'] = int(best_epoch)
    except Exception as e:
        print(f'Warning: resume failed, starting from scratch. Error: {e}')
else:
    print('Resume disabled. Training starts from epoch 1.')

=== CELL 8: Initialize W&B Run ===


best_acc_so_far,▁▇█████
best_epoch_so_far,▁▅█████
epoch,▁▂▃▅▆▇█
epochs_without_improvement,▁▁▁▁▃▆█
lr_main_max,████▃▃▁
lr_main_min,████▃▃▁
train_acc,▁▅▆▇▇██
train_knowledge_loss,█▅▄▃▂▁▁
train_l1,█▅▄▃▂▁▁
train_l2,█▅▄▃▂▁▁
+5,...


W&B run: https://wandb.ai/haidang262004-i-h-c-qu-c-gia-tp-hcm/transtr-causalvid-dino/runs/jxzvvatu
Resume enabled from: wandb


wandb: WARNING A graphql request initiated by the public wandb API timed out (timeout=19 sec). Create a new API with an integer timeout larger than 19, e.g., `api = wandb.Api(timeout=29)` to increase the graphql timeout.
wandb: Downloading large artifact 'last-checkpoint-gdino-nolora-ncod-hum:latest', 2507.81MB. 2 files...
wandb:   2 of 2 files downloaded.  
Done. 00:00:01.4 (1850.8MB/s)


Loaded history CSV with 7 rows
Resumed from: /content/tranSTR_Casual/artifacts/last-checkpoint-gdino-nolora-ncod-hum:v6/latest_checkpoint_gdino_nolora_ncod_hum.ckpt
Start epoch: 8 | Best acc: 69.69% | Best epoch: 3


In [ ]:
# CELL 9: Training Loop + Checkpoint/CSV Logging + Early Stopping
print('=== CELL 9: Training ===')

if RUN_TRAINING:
    stop_training = False
    for ep in range(start_epoch, args.epoch + 1):
        print(f'\nEpoch {ep}/{args.epoch}')
        total_loss, train_acc = train_epoch_qwen(
            model=model,
            optimizer=optimizer_model,
            loader=train_loader,
            ce_loss=ce_loss,
            device=device,
            epoch=ep,
            accumulation_steps=args.accumulation_steps
        )

        val_acc = eval_epoch_qwen(model, val_loader, device)
        scheduler.step(val_acc)

        history['train_loss'].append(total_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        current_lrs = [pg['lr'] for pg in optimizer_model.param_groups]
        min_lr_now = float(min(current_lrs))
        max_lr_now = float(max(current_lrs))

        improved = val_acc > (best_acc + args.early_stop_min_delta)
        if improved:
            best_acc = val_acc
            best_epoch = ep
            epochs_without_improvement = 0
            print(f'New best val_acc={best_acc:.2f}% at epoch {best_epoch}')
        elif ep >= args.early_stop_start_epoch:
            epochs_without_improvement += 1
            print(
                f'No significant improvement for {epochs_without_improvement} epoch(s) '
                f'(patience={args.early_stop_patience}, min_delta={args.early_stop_min_delta})'
            )

        epoch_row = {
            'epoch': ep,
            'train_total_loss': float(total_loss),
            'train_acc': float(train_acc),
            'val_acc': float(val_acc),
            'lr_main_min': min_lr_now,
            'lr_main_max': max_lr_now,
            'best_acc_so_far': float(best_acc),
            'best_epoch_so_far': int(best_epoch),
            'epochs_without_improvement': int(epochs_without_improvement)
        }
        history_rows.append(epoch_row)
        pd.DataFrame(history_rows).to_csv(TRAIN_HISTORY_CSV_PATH, index=False)

        wandb.log(epoch_row)

        ckpt = {
            'epoch': ep,
            'model_state_dict': model.state_dict(),
            'optimizer_model_state_dict': optimizer_model.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_acc': best_acc,
            'best_epoch': best_epoch,
            'epochs_without_improvement': epochs_without_improvement,
            'history': history,
            'history_rows': history_rows
        }

        torch.save(ckpt, LATEST_CKPT_PATH)

        last_artifact = wandb.Artifact(
            name=LAST_ARTIFACT_NAME,
            type='model',
            metadata={
                'epoch': ep,
                'best_acc': float(best_acc),
                'best_epoch': int(best_epoch),
                'val_acc': float(val_acc),
                'train_acc': float(train_acc),
                'train_total_loss': float(total_loss),
                'epochs_without_improvement': int(epochs_without_improvement)
            },
        )
        last_artifact.add_file(LATEST_CKPT_PATH, name=LATEST_CKPT_FILENAME)
        if os.path.exists(TRAIN_HISTORY_CSV_PATH):
            last_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name=TRAIN_HISTORY_FILENAME)
        wandb.log_artifact(last_artifact, aliases=['latest', f'epoch-{ep}'])

        if improved:
            torch.save(ckpt, save_path)
            best_artifact = wandb.Artifact(
                name=BEST_ARTIFACT_NAME,
                type='model',
                metadata={
                    'epoch': ep,
                    'best_acc': float(best_acc),
                    'val_acc': float(val_acc),
                    'train_acc': float(train_acc)
                },
            )
            best_artifact.add_file(save_path, name=MODEL_FILENAME)
            if os.path.exists(TRAIN_HISTORY_CSV_PATH):
                best_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name=TRAIN_HISTORY_FILENAME)
            wandb.log_artifact(best_artifact, aliases=['best', f'epoch-{ep}'])

        if ep >= args.early_stop_start_epoch and epochs_without_improvement >= args.early_stop_patience:
            print(f'Early stopping at epoch {ep}. Best val_acc={best_acc:.2f}% at epoch {best_epoch}.')
            wandb.run.summary['early_stopped'] = True
            wandb.run.summary['early_stop_epoch'] = int(ep)
            stop_training = True
            break

    wandb.run.summary['best_val_acc'] = float(best_acc)
    wandb.run.summary['best_epoch'] = int(best_epoch)

    if os.path.exists(save_path):
        best_ckpt = torch.load(save_path, map_location=device)
        model.load_state_dict(best_ckpt['model_state_dict'], strict=False)
        print(f'Loaded best checkpoint from epoch {best_epoch} for final evaluation.')

    if not stop_training:
        print(f'Training finished all {args.epoch} epochs. Best Val Accuracy: {best_acc:.2f}%')
else:
    print('Skipping training')

=== CELL 9: Training ===

Epoch 8/15


Epoch 8:   0%|          | 0/3519 [00:00<?, ?it/s]

No significant improvement for 4 epoch(s) (patience=4, min_delta=0.05)
Early stopping at epoch 8. Best val_acc=69.69% at epoch 3.
Loaded best checkpoint from epoch 3 for final evaluation.


In [ ]:
# CELL 10: Detailed Evaluation + Memory Post-check + CSV export
print('=== CELL 10: Detailed Evaluation + Memory Post-check ===')
import seaborn as sns
from networks.knowledge_retriever import CausalKnowledgeRetriever

CSV_OUTPUT_PATH = os.path.join(MODEL_DIR, PREDICTIONS_CSV_FILENAME)
COMPARISON_CSV_PATH = os.path.join(MODEL_DIR, 'run_comparison_qwen_minilm.csv')

# ===== Simple memory post-check config =====
TOPK_KNOWLEDGE = 5
MEMORY_PASS_THRESHOLD = 0.15
MEMORY_GATE_ENABLED = True
MEMORY_MARGIN = 0.05

def _resolve_kb_path():
    candidates = [
        os.path.join(os.getcwd(), 'data', 'causal_knowledge_bank.json'),
        '/content/tranSTR_Casual/causalvid/data/causal_knowledge_bank.json',
        '/kaggle/working/tranSTR_Casual/causalvid/data/causal_knowledge_bank.json',
        '/kaggle/working/causalvid/data/causal_knowledge_bank.json'
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

KB_PATH = _resolve_kb_path()
retriever = CausalKnowledgeRetriever(KB_PATH, topk=TOPK_KNOWLEDGE) if KB_PATH else None
print(f'Knowledge bank path: {KB_PATH if KB_PATH else "NOT FOUND (memory check disabled)"}')

def _qtype_to_family(qtype):
    qtype = str(qtype)
    valid = {'descriptive', 'explanatory', 'predictive', 'predictive_reason', 'counterfactual', 'counterfactual_reason'}
    return qtype if qtype in valid else 'descriptive'

def _score_candidate_with_memory(question, candidate, q_family, video_anchors=None):
    if retriever is None:
        return 0.0, []
    hits = retriever.retrieve(
        question=str(question),
        candidate=str(candidate),
        video_anchors=video_anchors or [],
        q_family=str(q_family),
        topk=TOPK_KNOWLEDGE
    )
    top_score = max([float(h.get('score', 0.0)) for h in hits], default=0.0)
    return top_score, hits

def _build_eval_meta_map(loader):
    dataset = getattr(loader, 'dataset', None)
    sample_list = getattr(dataset, 'sample_list', None) if dataset is not None else None
    meta_map = {}
    if sample_list is None:
        return meta_map

    for _, row in sample_list.iterrows():
        vid = str(row.get('video_id', ''))
        qtype = str(row.get('type', 'unknown'))
        qns_key = f'{vid}_{qtype}'
        meta_map[qns_key] = {
            'video_id': vid,
            'question_type': qtype,
            'question': str(row.get('question', '')),
            'answers': [str(row.get(f'a{i}', '')) for i in range(5)]
        }
    return meta_map

def evaluate_detailed_v2(model, loader, device, log_to_wandb=True):
    model.eval()
    type_results = {}
    prediction_rows = []
    meta_map = _build_eval_meta_map(loader)

    memory_match_flags = []
    memory_pass_flags = []
    memory_gate_correct_flags = []

    with torch.no_grad():
        for batch in tqdm(loader):
            ff, of, qns, ans_word, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of = ff.to(device), of.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None

            out = model(ff, of, qns, ans_word, return_aux=True, q_family_id=q_family_id)
            logits = out['qwen_logits']
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            preds = probs.argmax(axis=-1)
            targets = ans_id.numpy()

            for i, (key, pred, target, prob_vec) in enumerate(zip(qns_keys, preds, targets, probs)):
                meta = meta_map.get(str(key), {})
                qtype = str(meta.get('question_type', 'unknown'))
                q_family = _qtype_to_family(qtype)
                video_id = str(meta.get('video_id', str(key)))
                question = str(meta.get('question', qns[i]))
                answers = meta.get('answers', ['', '', '', '', ''])
                if len(answers) < 5:
                    answers += [''] * (5 - len(answers))
                answers = answers[:5]

                correct_idx = int(target)
                predicted_idx = int(pred)
                is_correct = int(correct_idx == predicted_idx)

                # Post-check prediction against memory support
                cand_mem_scores = []
                for cand in answers:
                    score, _ = _score_candidate_with_memory(question, cand, q_family, video_anchors=[video_id])
                    cand_mem_scores.append(float(score))

                mem_best_idx = int(np.argmax(cand_mem_scores)) if len(cand_mem_scores) > 0 else predicted_idx
                pred_mem_score = float(cand_mem_scores[predicted_idx]) if len(cand_mem_scores) > predicted_idx else 0.0
                gt_mem_score = float(cand_mem_scores[correct_idx]) if len(cand_mem_scores) > correct_idx else 0.0

                memory_match_pred = int(predicted_idx == mem_best_idx)
                memory_pass_pred = int(pred_mem_score >= MEMORY_PASS_THRESHOLD)

                gated_idx = predicted_idx
                if MEMORY_GATE_ENABLED and len(cand_mem_scores) > 0:
                    if (cand_mem_scores[mem_best_idx] - pred_mem_score) >= MEMORY_MARGIN:
                        gated_idx = mem_best_idx
                gated_correct = int(gated_idx == correct_idx)

                memory_match_flags.append(memory_match_pred)
                memory_pass_flags.append(memory_pass_pred)
                memory_gate_correct_flags.append(gated_correct)

                type_results.setdefault(qtype, []).append({
                    'video_id': video_id,
                    'pred': predicted_idx,
                    'target': correct_idx,
                    'correct': bool(is_correct)
                })

                prediction_rows.append({
                    'video_id': video_id,
                    'question_type': qtype,
                    'question': question,
                    'correct_idx': correct_idx,
                    'predicted_idx': predicted_idx,
                    'predicted_idx_after_memory_gate': int(gated_idx),
                    'is_correct': is_correct,
                    'is_correct_after_memory_gate': gated_correct,
                    'confidence': float(prob_vec[predicted_idx]),
                    'memory_best_idx': mem_best_idx,
                    'memory_match_pred': memory_match_pred,
                    'memory_pass_pred': memory_pass_pred,
                    'memory_pred_score': pred_mem_score,
                    'memory_gt_score': gt_mem_score,
                    'a0': answers[0],
                    'prob_a0': float(prob_vec[0]),
                    'a1': answers[1],
                    'prob_a1': float(prob_vec[1]),
                    'a2': answers[2],
                    'prob_a2': float(prob_vec[2]),
                    'a3': answers[3],
                    'prob_a3': float(prob_vec[3]),
                    'a4': answers[4],
                    'prob_a4': float(prob_vec[4]),
                    'predicted_answer': answers[predicted_idx],
                    'correct_answer': answers[correct_idx]
                })

    prediction_df = pd.DataFrame(prediction_rows)
    prediction_df.to_csv(CSV_OUTPUT_PATH, index=False)
    print(f'Saved detailed predictions CSV: {CSV_OUTPUT_PATH}')

    metrics = {}
    mapping = [
        ('Description', 'descriptive'),
        ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'),
        ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason')
    ]
    for name, qtype in mapping:
        rows = type_results.get(qtype, [])
        total = len(rows)
        correct = sum(1 for r in rows if r['correct'])
        metrics[name] = (correct / total * 100) if total > 0 else 0.0

    def _calc_hard_metric(type_ans, type_reason):
        if type_ans not in type_results or type_reason not in type_results:
            return 0.0
        ans_by_vid = {r['video_id']: r['correct'] for r in type_results[type_ans]}
        reason_by_vid = {r['video_id']: r['correct'] for r in type_results[type_reason]}
        common_vids = set(ans_by_vid.keys()) & set(reason_by_vid.keys())
        if len(common_vids) == 0:
            return 0.0
        both_correct = sum(1 for vid in common_vids if ans_by_vid[vid] and reason_by_vid[vid])
        return both_correct / len(common_vids) * 100

    metrics['PAR'] = _calc_hard_metric('predictive', 'predictive_reason')
    metrics['CAR'] = _calc_hard_metric('counterfactual', 'counterfactual_reason')
    metrics['Acc_ALL'] = (
        metrics['Description'] + metrics['Explanation'] + metrics['PAR'] + metrics['CAR']
    ) / 4.0

    # New memory post-check metrics
    metrics['Memory_Consistency'] = float(np.mean(memory_match_flags) * 100) if len(memory_match_flags) > 0 else 0.0
    metrics['Memory_Pass_Rate'] = float(np.mean(memory_pass_flags) * 100) if len(memory_pass_flags) > 0 else 0.0
    metrics['Memory_Gated_Acc'] = float(np.mean(memory_gate_correct_flags) * 100) if len(memory_gate_correct_flags) > 0 else 0.0

    # Weighted score for run selection (weak-group priority)
    metrics['WeightedScore_WeakPriority'] = (
        0.35 * metrics['Predictive-Reason'] +
        0.35 * metrics['Counterfactual-Reason'] +
        0.20 * metrics['Acc_ALL'] +
        0.10 * float(best_acc)
    )

    if log_to_wandb:
        wandb.log({
            'eval/Description': metrics['Description'],
            'eval/Explanation': metrics['Explanation'],
            'eval/Predictive_Answer': metrics['Predictive-Answer'],
            'eval/Predictive_Reason': metrics['Predictive-Reason'],
            'eval/Counterfactual_Answer': metrics['Counterfactual-Answer'],
            'eval/Counterfactual_Reason': metrics['Counterfactual-Reason'],
            'eval/PAR': metrics['PAR'],
            'eval/CAR': metrics['CAR'],
            'eval/Acc_ALL': metrics['Acc_ALL'],
            'eval/Memory_Consistency': metrics['Memory_Consistency'],
            'eval/Memory_Pass_Rate': metrics['Memory_Pass_Rate'],
            'eval/Memory_Gated_Acc': metrics['Memory_Gated_Acc'],
            'eval/WeightedScore_WeakPriority': metrics['WeightedScore_WeakPriority']
        })

    print(f"PAR: {metrics['PAR']:.2f}% | CAR: {metrics['CAR']:.2f}% | Acc_ALL: {metrics['Acc_ALL']:.2f}%")
    print(f"Memory_Consistency: {metrics['Memory_Consistency']:.2f}% | Memory_Gated_Acc: {metrics['Memory_Gated_Acc']:.2f}%")
    print(f"WeightedScore_WeakPriority: {metrics['WeightedScore_WeakPriority']:.2f}")
    return metrics, type_results, CSV_OUTPUT_PATH

metrics, raw_results, predictions_csv = evaluate_detailed_v2(model, test_loader, device, log_to_wandb=True)

# Append/update run comparison table
comparison_row = {
    'run_tag': RUN_TAG,
    'run_profile': RUN_PROFILE,
    'best_val_acc': float(best_acc),
    'best_epoch': int(best_epoch),
    'Description': float(metrics.get('Description', 0.0)),
    'Explanation': float(metrics.get('Explanation', 0.0)),
    'Predictive-Answer': float(metrics.get('Predictive-Answer', 0.0)),
    'Predictive-Reason': float(metrics.get('Predictive-Reason', 0.0)),
    'Counterfactual-Answer': float(metrics.get('Counterfactual-Answer', 0.0)),
    'Counterfactual-Reason': float(metrics.get('Counterfactual-Reason', 0.0)),
    'PAR': float(metrics.get('PAR', 0.0)),
    'CAR': float(metrics.get('CAR', 0.0)),
    'Acc_ALL': float(metrics.get('Acc_ALL', 0.0)),
    'WeightedScore_WeakPriority': float(metrics.get('WeightedScore_WeakPriority', 0.0)),
}

if os.path.exists(COMPARISON_CSV_PATH):
    comp_df = pd.read_csv(COMPARISON_CSV_PATH)
    comp_df = comp_df[comp_df['run_tag'] != RUN_TAG]
    comp_df = pd.concat([comp_df, pd.DataFrame([comparison_row])], ignore_index=True)
else:
    comp_df = pd.DataFrame([comparison_row])

comp_df = comp_df.sort_values(by='WeightedScore_WeakPriority', ascending=False)
comp_df.to_csv(COMPARISON_CSV_PATH, index=False)
print(f'Saved/updated run comparison CSV: {COMPARISON_CSV_PATH}')
print(comp_df[['run_tag', 'run_profile', 'best_val_acc', 'Predictive-Reason', 'Counterfactual-Reason', 'Acc_ALL', 'WeightedScore_WeakPriority']])

wandb.run.summary['run_tag'] = RUN_TAG
wandb.run.summary['run_profile'] = RUN_PROFILE
wandb.run.summary['weighted_score_weak_priority'] = float(metrics['WeightedScore_WeakPriority'])
print(metrics)

=== CELL 10: Detailed Evaluation + Memory Post-check ===
Knowledge bank path: /content/tranSTR_Casual/data/causal_knowledge_bank.json


  0%|          | 0/1018 [00:00<?, ?it/s]

Saved detailed predictions CSV: /content/working/models/test_predictions_gdino_nolora_ncod_hum.csv
PAR: 50.28% | CAR: 50.39% | Acc_ALL: 62.90%
Memory_Consistency: 20.11% | Memory_Gated_Acc: 68.55%
{'Description': 73.38739402875046, 'Explanation': 77.53409509767785, 'Predictive-Answer': 65.35200884629562, 'Predictive-Reason': 65.94176188720972, 'Counterfactual-Answer': 68.63251013638039, 'Counterfactual-Reason': 64.5963877626244, 'PAR': 50.276446737928495, 'CAR': 50.387025433099886, 'Acc_ALL': 62.89624032436417, 'Memory_Consistency': 20.10689273866568, 'Memory_Pass_Rate': 66.79567514436663, 'Memory_Gated_Acc': 68.54957611500184}


In [ ]:
# CELL 11: Finish W&B
print('=== CELL 11: Finish W&B ===')

METRICS_JSON_PATH = os.path.join(MODEL_DIR, METRICS_JSON_FILENAME)
with open(METRICS_JSON_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Saved metrics JSON: {METRICS_JSON_PATH}')

UPLOAD_CKPT_ARTIFACTS_AT_FINISH = True
if UPLOAD_CKPT_ARTIFACTS_AT_FINISH:
    if os.path.exists(LATEST_CKPT_PATH):
        latest_ckpt_artifact = wandb.Artifact(
            name=LAST_ARTIFACT_NAME,
            type='model',
            metadata={
                'stage': 'finish',
                'checkpoint_kind': 'latest',
                'run_tag': RUN_TAG,
                'run_profile': RUN_PROFILE,
                'minilm_name': args.minilm_name,
                'qwen_name': args.qwen_name,
                'qwen_lora': args.use_qwen_lora
            }
        )
        latest_ckpt_artifact.add_file(LATEST_CKPT_PATH, name=LATEST_CKPT_FILENAME)
        if os.path.exists(TRAIN_HISTORY_CSV_PATH):
            latest_ckpt_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name=TRAIN_HISTORY_FILENAME)
        wandb.log_artifact(latest_ckpt_artifact, aliases=['latest', 'finish'])
        print('Uploaded latest checkpoint artifact to W&B.')
    else:
        print(f'Warning: latest checkpoint not found at {LATEST_CKPT_PATH}')

    if os.path.exists(save_path):
        best_ckpt_artifact = wandb.Artifact(
            name=BEST_ARTIFACT_NAME,
            type='model',
            metadata={
                'stage': 'finish',
                'checkpoint_kind': 'best',
                'run_tag': RUN_TAG,
                'run_profile': RUN_PROFILE,
                'minilm_name': args.minilm_name,
                'qwen_name': args.qwen_name,
                'qwen_lora': args.use_qwen_lora
            }
        )
        best_ckpt_artifact.add_file(save_path, name=MODEL_FILENAME)
        if os.path.exists(TRAIN_HISTORY_CSV_PATH):
            best_ckpt_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name=TRAIN_HISTORY_FILENAME)
        wandb.log_artifact(best_ckpt_artifact, aliases=['best', 'finish'])
        print('Uploaded best checkpoint artifact to W&B.')
    else:
        print(f'Warning: best checkpoint not found at {save_path}')

final_artifact = wandb.Artifact(
    name=FINAL_ARTIFACT_NAME,
    type='results',
    metadata={
        'run_tag': RUN_TAG,
        'run_profile': RUN_PROFILE,
        'backbone': 'dinov3+groundingdino',
        'minilm_name': args.minilm_name,
        'qwen_name': args.qwen_name,
        'qwen_lora': args.use_qwen_lora
    }
)
if os.path.exists(METRICS_JSON_PATH):
    final_artifact.add_file(METRICS_JSON_PATH, name=METRICS_JSON_FILENAME)
if os.path.exists(predictions_csv):
    final_artifact.add_file(predictions_csv, name=PREDICTIONS_CSV_FILENAME)
if os.path.exists(TRAIN_HISTORY_CSV_PATH):
    final_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name=TRAIN_HISTORY_FILENAME)
if os.path.exists(LATEST_CKPT_PATH):
    final_artifact.add_file(LATEST_CKPT_PATH, name=LATEST_CKPT_FILENAME)
if os.path.exists(save_path):
    final_artifact.add_file(save_path, name=MODEL_FILENAME)

comparison_csv_path = os.path.join(MODEL_DIR, 'run_comparison_qwen_minilm.csv')
if os.path.exists(comparison_csv_path):
    final_artifact.add_file(comparison_csv_path, name='run_comparison_qwen_minilm.csv')

wandb.log_artifact(final_artifact)
wandb.finish()
print('W&B run finished with metrics, CSVs, and checkpoints artifacts.')

=== CELL 11: Finish W&B ===
Saved metrics JSON: /content/working/models/final_metrics_gdino_nolora_ncod_hum.json
Uploaded latest checkpoint artifact to W&B.


wandb: WARNING Artifact "last-checkpoint-gdino-nolora-ncod-hum" already exists with the same content. No new version will be created.


Uploaded best checkpoint artifact to W&B.


best_acc_so_far,▁
best_epoch_so_far,▁
epoch,▁
epochs_without_improvement,▁
eval/Acc_ALL,▁
eval/CAR,▁
eval/Counterfactual_Answer,▁
eval/Counterfactual_Reason,▁
eval/Description,▁
eval/Explanation,▁
+17,...


W&B run finished with metrics, CSVs, and checkpoints artifacts.
